# Parity Angle Experiment — Analysis

Assign the RIDs you care about; for each run you get the **QWP-vs-visibility
sinusoidal fit** (best QWP from the fit), the **best-QWP parity oscillation**, the
**bar splitting** at the optimal HWP, and how the optimal **QWP / HWP change with
basis angle**.

Thin driver over `analysis/analyze_parity_angle.py`; `%autoreload` picks up edits.

> **Run in the working-BLAS `python` env, NOT `artiq7_claude`.**

## 0. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os, glob
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ANALYSIS_DIR = Path(r"C:\Networking Experiment\Artiq7_TEST_NEW\useful_codes\repository\analysis")
for p in (str(ANALYSIS_DIR.parent), str(ANALYSIS_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

from analyze_parity_angle import (
    find_runs, read_basis_phase, analyze_run, print_summary,
    plot_qwp_slice, plot_visibility_summary,
    analyze_run_set, print_run_set_table,
    plot_run_set, plot_run_set_slices, plot_run_set_bars,
    plot_bar_split, plot_waveplate_vs_phase,
    plot_qwp_fit, plot_run_set_qwpfits,
    RESULTS_DIR,
)

%matplotlib inline
print("Setup complete - functions from analyze_parity_angle.py")
print(f"Results dir: {RESULTS_DIR}")

## 1. List Available ParityAngleExperiment Runs

In [ ]:
files = find_runs()
if not files:
    print(f"No ParityAngleExperiment .h5 under {RESULTS_DIR}")
else:
    print(f"{'RID':>9s}  {'phase_deg':>9s}  File")
    print("-" * 60)
    for fp in files:
        rid = os.path.basename(fp).split("-")[0]
        ph = read_basis_phase(fp)
        print(f"{rid:>9s}  {ph:>9.2f}  {os.path.basename(fp)}")

## 2. Assign your RID set  ← main input

Put the RIDs you want in `RID_SET`. Missing angles are skipped automatically.

- **best QWP** now comes from a **sinusoidal fit** of visibility vs QWP
  (continuous), not the discrete experimental max.
- **HWP_opt** comes from the cos⁴ parity fit.
- **PARITY_CONVENTION** fixes the bar pattern so it never flips: `"plus"` →
  `[0, .5, .5, 0]`, `"minus"` → `[.5, 0, 0, .5]`.

In [ ]:
########################################
RID_SET           = [1135, 1138, 1141, 1144, 1147, 1150]   # <-- your runs
THRES             = None      # atom threshold, or None = file value
PARITY_CONVENTION = "plus"    # "plus" -> [0,.5,.5,0] ; "minus" -> [.5,0,0,.5]
########################################

run_set = analyze_run_set(RID_SET, thres=THRES, parity_convention=PARITY_CONVENTION)
print_run_set_table(run_set)

## 3. QWP vs visibility — sinusoidal fit (per RID)

For each RID, parity visibility vs QWP with the sinusoidal fit; the fitted best
QWP (green line) is what feeds the table above.

In [ ]:
plot_run_set_qwpfits(run_set)

## 4. Best-QWP parity oscillation for each RID  (HWP_opt marked)

In [ ]:
plot_run_set_slices(run_set)

## 5. Bar splitting at the optimal HWP  (per RID)

Four category probabilities at HWP_opt. With PARITY_CONVENTION fixed, the split
pattern is the same for every RID (no flipping).

In [ ]:
plot_run_set_bars(run_set)

## 6. Optimal QWP / HWP vs basis angle

In [ ]:
plot_waveplate_vs_phase(run_set)

## 7. Maximum visibility vs basis angle

In [ ]:
plot_run_set(run_set)

## 8. Single-Run Deep Dive (optional)

Analyze one run on its own — QWP fit + every QWP slice.

In [ ]:
########################################
RID   = None    # int RID, or None for latest
THRES = None
PARITY_CONVENTION = "plus"
########################################
files = find_runs()
if RID is not None:
    filepath = [f for f in files if os.path.basename(f).split("-")[0] == f"{RID:09d}"][-1]
else:
    filepath = files[-1]; print("(latest run)")
meta, analyzed, df = analyze_run(filepath, thres=THRES, parity_convention=PARITY_CONVENTION)
best_qwp = print_summary(meta, analyzed)
plot_qwp_fit(meta)
for qv in sorted(analyzed):
    plot_qwp_slice(meta, qv, analyzed[qv])
    plot_bar_split(meta, qv, analyzed[qv])
plot_visibility_summary(meta, analyzed)